In [1]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import IntegerType
from pyspark.sql.types import StructType, StructField
import pandas as pd

# Initialize Spark session
spark = SparkSession.builder \
    .appName("VectorizedUDFDemo") \
    .getOrCreate()

In [2]:
# Step 1: Create a sample DataFrame
data = [
    (1, 100),
    (2, 200),
    (3, 300),
    (4, 400),
    (5, 500)
]

columns = ["id", "value"]

df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show()

Original DataFrame:
+---+-----+
| id|value|
+---+-----+
|  1|  100|
|  2|  200|
|  3|  300|
|  4|  400|
|  5|  500|
+---+-----+



In [3]:
# Step 2: Define a Pandas UDF for multiple outputs: sum and product
@pandas_udf("struct<sum:int, product:int>")
def calculate_sum_and_product(col1: pd.Series, col2: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        "sum": col1 + col2,
        "product": col1 * col2
    })

In [ ]:
# Apply the UDF to calculate 'sum' and 'product'
df_with_results = df.withColumn("results", calculate_sum_and_product(df["id"], df["value"]))

# Extract 'sum' and 'product' from the struct column
df_final = df_with_results.select(
    "id", "value",
    df_with_results["results.sum"].alias("sum"),
    df_with_results["results.product"].alias("product")
)

print("DataFrame After Applying Advanced Vectorized UDF:")
df_final.show()

DataFrame After Applying Advanced Vectorized UDF:
